# 06 · Modeling

**Stage 6 of 8** · ACCESS → WRANGLE → EXPLORE → VISUALIZE → ENGINEER → MODEL → EVALUATE → RECOMMEND

> **Project premise:** if someone has had cancer and was treated, **will it come back?** And for every factor linked to it coming back: **correlation or causation?**
>
> All data here is public: a k-anonymous feature table (every combination of site group, region, sex and diagnosis period is shared by ≥ 3 patients), synthetic patients, and aggregate reports. The raw database and clinical notes stay private.

**Goal of this stage:** build models that estimate each patient's risk of their cancer coming back, using only pre-relapse information.

**Business question:** Can we rank treated patients by risk so that follow-up goes to the highest-risk patients first?

In [1]:
%matplotlib inline
import sys, json, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
PUBLIC = ROOT / "data" / "public" / "patients_features_public.csv"
SYNTH = ROOT / "data" / "synthetic" / "synthetic_patients.csv"
BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8984"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e5e0", "axes.axisbelow": True, "figure.dpi": 110})
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 160)
df = pd.read_csv(PUBLIC)

### Q1 · Target
`relapse` (1 = cancer came back), with `rfs_years` (time) for the survival model. 138 at-risk patients, 30 relapses.

### Q2–Q3 · Features (X) and target (y)

In [2]:
import model as M
X, y = M.load(PUBLIC)
print("X:", X.shape, "| y:", y.shape, "| relapse rate:", f"{y.mean():.0%}")
X.head()

X: (138, 15) | y: (138,) | relapse rate: 22%


,site_group,breast_subtype,region,years_since_dx,grade_known,grade3,surgery,radiotherapy,hormonal_therapy,neoadjuvant_chemo,adjuvant_chemo,anti_her2,treated_abroad,missed_followup_or_delay,locally_advanced
1,Genitourinary,not breast,Suppressed,9.0,0,0,1,0,0,0,0,0,1,0,0
2,Skin,not breast,Suppressed,6.0,0,0,0,1,0,0,0,0,0,0,0
5,Breast,Unknown,Khartoum,8.0,0,0,1,1,1,0,0,0,0,0,0
8,Breast,Triple negative,Khartoum,7.0,1,1,1,0,1,1,0,0,0,0,0
11,Gynaecologic,not breast,Suppressed,5.0,0,0,0,0,0,0,0,0,0,0,0


### Q4 · Why not a single train/test split?
With 30 relapses, a 20% test set would contain only ~6 relapses, and the score would depend on luck. Instead:
**stratified 5-fold cross-validation, repeated 20 times**. Every patient is tested on a model that never saw them,
and we report the average and spread.

### Q5 · What kind of problem is this?
Two framings:
- **Classification** (came back yes/no): logistic regression, random forest.
- **Survival** (how long until it comes back): Cox proportional-hazards model. This is the better fit, because
  patients were followed 1 to 10+ years and "no relapse yet" depends on how long we've looked.

### Q6–Q7 · Train the models and predict on unseen folds

In [3]:
res, summary, oof, coefs = M.cross_validate(X, y)
summary.xs("mean", axis=1, level=1)[["roc_auc", "accuracy", "precision", "recall", "f1"]].round(3)

,roc_auc,accuracy,precision,recall,f1
model,,,,,
Baseline (random flagging),0.492,0.648,0.190,0.190,0.190
Logistic regression,0.475,0.646,0.187,0.187,0.187
Random forest,0.588,0.728,0.375,0.375,0.375


In [4]:
cox = M.cox_cv(PUBLIC)
pd.DataFrame(cox).T

,c_index_mean,c_index_sd
Cox: site + grade + treatments,0.629,0.027
Cox: breast vs other only,0.593,0.024


**Reading the results**
- The yes/no models are **close to a coin flip** (AUC ≈ 0.5–0.6). They can't distinguish "didn't relapse" from
  "not followed long enough".
- The time-aware **Cox model reaches a C-index of about 0.63**, and cancer type alone gets about 0.59. So most of the
  (weak) signal is cancer type.

### What the logistic model leans on (associations, not causes)

In [5]:
c = coefs.mean(axis=1).sort_values()
c.index = [i.split("__", 1)[1] for i in c.index]
pd.concat([c.head(5), c.tail(5)]).round(2).to_frame("mean coefficient (+ = more risk)")

,mean coefficient (+ = more risk)
radiotherapy,-0.65
breast_subtype_HR+/HER2-,-0.59
site_group_CNS,-0.50
region_Khartoum,-0.41
site_group_Skin,-0.41
region_Suppressed,0.39
site_group_Sarcoma,0.44
breast_subtype_HER2+,0.52
site_group_Head & neck,0.56
adjuvant_chemo,0.60


## So what?

With this data, relapse is **only weakly predictable**. That's a result, not a failure: it tells the clinic that
risk can't be estimated without the missing variables (stage, age, receptor status).